In [1]:
!python --version


Python 3.12.13


In [2]:
from datasets import load_dataset
import pandas as pd

In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

In [4]:
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: NVIDIA GeForce RTX 4050 Laptop GPU


In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)

cuda


In [6]:
model_name =model_name = "roberta-base"

In [7]:
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

In [8]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=1
)
model.to(device)

# saves memory, lets you use a bigger batch size — worth it since deberta-v3-small is bigger than distilbert
model.gradient_checkpointing_enable()

print("roberta-base reward model ready")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.bias         | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


roberta-base reward model ready


In [9]:
train = pd.read_csv("../preprocess/train.csv")
validation = pd.read_csv("../preprocess/validation.csv")
test = pd.read_csv("../preprocess/test.csv")

print("Train:", train.shape)
print("Validation:", validation.shape)
print("Test:", test.shape)

Train: (31740, 4)
Validation: (3968, 4)
Test: (3968, 4)


In [10]:
print(train.columns)
print(train.head())

Index(['prompt', 'response_a', 'response_b', 'label'], dtype='str')
                                              prompt  \
0  User: Will FDVR tech be possible in the future...   
1  User: Tell me what would be good antenna for r...   
2            User: what is 100 days after november 1   
3     User: What is the best hotel in San Francisco?   
4  User: Traduit en francais: Outcome questionnai...   

                                          response_a  \
0  FDVR stands for Foveated Direct Voxel Renderin...   
1  There are several factors to consider when des...   
2  100 days after November 1, 2023, would be Febr...   
3  There are several highly-rated hotels in San F...   
4  Sure, I'll do my best to assist you with care,...   

                                          response_b  label  
0  As an AI assistant, I can make an educated gue...      1  
1  The best antenna for a rocket going to GSO wou...      0  
2  To find out what day it is 100 days after Nove...      0  
3  The bes

In [11]:
print(train["label"].value_counts())

label
1    15946
0    15794
Name: count, dtype: int64


In [12]:
print(train["label"].value_counts(normalize=True))

label
1    0.502394
0    0.497606
Name: proportion, dtype: float64


In [13]:
print(tokenizer)

RobertaTokenizer(name_or_path='roberta-base', vocab_size=50265, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'sep_token': '</s>', 'pad_token': '<pad>', 'cls_token': '<s>', 'mask_token': '<mask>'}, added_tokens_decoder={
	0: AddedToken("<s>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
	1: AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
	2: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
	3: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
	50264: AddedToken("<mask>", rstrip=False, lstrip=True, single_word=False, normalized=True, special=True),
})


In [14]:
print(train[["prompt", "response_a", "response_b"]].isna().sum())

prompt        0
response_a    4
response_b    4
dtype: int64


In [15]:
train = train.dropna(subset=["prompt", "response_a", "response_b"])
validation = validation.dropna(subset=["prompt", "response_a", "response_b"])
test = test.dropna(subset=["prompt", "response_a", "response_b"])

In [16]:
def make_text(row):
    return (
        "Prompt: " + row["prompt"] +
        "\nResponse A: " + row["response_a"] +
        "\nResponse B: " + row["response_b"]
    )

train["text"] = train.apply(make_text, axis=1)
validation["text"] = validation.apply(make_text, axis=1)
test["text"] = test.apply(make_text, axis=1)

print("Text created successfully")

Text created successfully


In [17]:
from datasets import Dataset

train_ds = Dataset.from_pandas(train)
val_ds = Dataset.from_pandas(validation)
test_ds = Dataset.from_pandas(test)

print(train_ds)

Dataset({
    features: ['prompt', 'response_a', 'response_b', 'label', 'text', '__index_level_0__'],
    num_rows: 31736
})


In [18]:
def create_inputs(example):
    prompt = str(example["prompt"])

    text_a = prompt + "\nResponse: " + str(example["response_a"])
    text_b = prompt + "\nResponse: " + str(example["response_b"])

    return {
        "text_a": text_a,
        "text_b": text_b,
        "label": example["label"]
    }

train_ds = train_ds.map(create_inputs)
val_ds = val_ds.map(create_inputs)
test_ds = test_ds.map(create_inputs)

Map:   0%|          | 0/31736 [00:00<?, ? examples/s]

Map:   0%|          | 0/3966 [00:00<?, ? examples/s]

Map:   0%|          | 0/3966 [00:00<?, ? examples/s]

In [19]:
def tokenize_data(example):
    tokens_a = tokenizer(
        example["text_a"],
        truncation=True,
        max_length=256
    )

    tokens_b = tokenizer(
        example["text_b"],
        truncation=True,
        max_length=256
    )

    return {
        "input_ids_a": tokens_a["input_ids"],
        "attention_mask_a": tokens_a["attention_mask"],
        "input_ids_b": tokens_b["input_ids"],
        "attention_mask_b": tokens_b["attention_mask"],
        "label": example["label"]
    }

In [20]:
train_ds = train_ds.map(tokenize_data)
val_ds = val_ds.map(tokenize_data)
test_ds = test_ds.map(tokenize_data)

Map:   0%|          | 0/31736 [00:00<?, ? examples/s]

Map:   0%|          | 0/3966 [00:00<?, ? examples/s]

Map:   0%|          | 0/3966 [00:00<?, ? examples/s]

In [23]:
print(max(train_ds["input_ids_a"][0]))

50118


In [24]:
def load_fresh_model():
    m = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=1)
    m.to(device)
    m.gradient_checkpointing_enable()
    return m

model = load_fresh_model()
print("Fresh model loaded, dtype:", next(model.parameters()).dtype)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.bias         | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Fresh model loaded, dtype: torch.float32


In [25]:
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

epochs = 4
batch_size = 16
grad_accum_steps = 2

optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)

steps_per_epoch = len(train_ds) // (batch_size * grad_accum_steps)
total_steps = steps_per_epoch * epochs
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.06 * total_steps),
    num_training_steps=total_steps
)

In [26]:
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence

def collate_fn(batch):
    input_ids = (
        [torch.tensor(x["input_ids_a"]) for x in batch] +
        [torch.tensor(x["input_ids_b"]) for x in batch]
    )
    attention_mask = (
        [torch.tensor(x["attention_mask_a"]) for x in batch] +
        [torch.tensor(x["attention_mask_b"]) for x in batch]
    )

    input_ids = pad_sequence(input_ids, batch_first=True, padding_value=tokenizer.pad_token_id)
    attention_mask = pad_sequence(attention_mask, batch_first=True, padding_value=0)

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "label": torch.tensor([x["label"] for x in batch]),
        "n": len(batch)
    }

train_loader = DataLoader(
    train_ds,               
    batch_size=batch_size,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=0,
    pin_memory=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=batch_size,
    collate_fn=collate_fn,
    num_workers=0,
    pin_memory=True
)

In [27]:
from torch.amp import autocast
from torch.nn.functional import logsigmoid

In [29]:
model = load_fresh_model()

epochs = 4
batch_size = 16
grad_accum_steps = 2

optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)

steps_per_epoch = len(train_ds) // (batch_size * grad_accum_steps)
total_steps = steps_per_epoch * epochs
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.06 * total_steps),
    num_training_steps=total_steps
)

for epoch in range(epochs):
    model.train()
    total_loss = 0
    optimizer.zero_grad()

    for step, batch in enumerate(train_loader):
        n = batch["n"]
        input_ids = batch["input_ids"].to(device, non_blocking=True)
        attention_mask = batch["attention_mask"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)

        with autocast("cuda", dtype=torch.bfloat16):
            logits = model(input_ids=input_ids, attention_mask=attention_mask).logits.squeeze(-1)
            reward_a = logits[:n]
            reward_b = logits[n:]

            preferred = torch.where(labels == 0, reward_a, reward_b)
            rejected = torch.where(labels == 0, reward_b, reward_a)

            loss = -logsigmoid(preferred - rejected).mean() / grad_accum_steps

        if torch.isnan(loss):
            print(f"NaN at epoch {epoch+1} step {step} — stopping.")
            break

        loss.backward()

        if (step + 1) % grad_accum_steps == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

        total_loss += loss.item() * grad_accum_steps

    print(f"Epoch: {epoch+1}  Loss: {total_loss / len(train_loader):.4f}")

    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for batch in val_loader:
            n = batch["n"]
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)

            with autocast("cuda", dtype=torch.bfloat16):
                logits = model(input_ids=input_ids, attention_mask=attention_mask).logits.squeeze(-1)

            reward_a, reward_b = logits[:n], logits[n:]
            predictions = (reward_b > reward_a).long()
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    val_acc = correct / total
    print(f"  Val Accuracy: {val_acc*100:.2f}%")

    model.save_pretrained(f"./roberta_reward_model_epoch{epoch+1}")
    tokenizer.save_pretrained(f"./roberta_reward_model_epoch{epoch+1}")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.bias         | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch: 1  Loss: 0.6470
  Val Accuracy: 63.99%


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch: 2  Loss: 0.6057
  Val Accuracy: 64.42%


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch: 3  Loss: 0.5427
  Val Accuracy: 63.64%


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch: 4  Loss: 0.4670
  Val Accuracy: 62.68%


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [34]:
model = AutoModelForSequenceClassification.from_pretrained("./roberta_reward_model_epoch2")
tokenizer = AutoTokenizer.from_pretrained("./roberta_reward_model_epoch2")
model.to(device)
model.eval()

print("Loaded epoch 2 checkpoint (best val accuracy: 64.42%)")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loaded epoch 2 checkpoint (best val accuracy: 64.42%)


In [35]:
def predict_preference(prompt, response_a, response_b):
    model.eval()
    text_a = f"{prompt}\nResponse: {response_a}"
    text_b = f"{prompt}\nResponse: {response_b}"

    inputs_a = tokenizer(text_a, return_tensors="pt", truncation=True, max_length=256).to(device)
    inputs_b = tokenizer(text_b, return_tensors="pt", truncation=True, max_length=256).to(device)

    with torch.no_grad():
        reward_a = model(**inputs_a).logits.squeeze().item()
        reward_b = model(**inputs_b).logits.squeeze().item()

    probability_a = torch.sigmoid(torch.tensor(reward_a - reward_b)).item()
    probability_b = 1 - probability_a
    preference = "Response A" if probability_a >= probability_b else "Response B"

    return {"reward_a": reward_a, "reward_b": reward_b,
            "probability_a": probability_a, "probability_b": probability_b,
            "preference": preference}

In [36]:
result = predict_preference(
    "Explain machine learning to a beginner.",
    
    "Machine learning is a way for computers to learn patterns from data and make predictions without being explicitly programmed for every task.",
    
    "Machine learning is when machines learn."
)

print(f"Response A reward: {result['reward_a']:.4f}")
print(f"Response B reward: {result['reward_b']:.4f}")

print(f"Response A probability: {result['probability_a'] * 100:.2f}%")
print(f"Response B probability: {result['probability_b'] * 100:.2f}%")

print(f"Predicted preference: {result['preference']}")

Response A reward: -2.5715
Response B reward: -2.8722
Response A probability: 57.46%
Response B probability: 42.54%
Predicted preference: Response A


In [40]:
test_loader = DataLoader(test_ds, batch_size=16, collate_fn=collate_fn, num_workers=0)

model.eval()
correct, total = 0, 0
with torch.no_grad():
    for batch in test_loader:
        n = batch["n"]
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)
        with autocast("cuda", dtype=torch.bfloat16):
            logits = model(input_ids=input_ids, attention_mask=attention_mask).logits.squeeze(-1)
        reward_a, reward_b = logits[:n], logits[n:]
        predictions = (reward_b > reward_a).long()
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

print(f"Test Accuracy: {correct/total*100:.2f}%")

Test Accuracy: 64.22%


In [41]:
import shutil
shutil.copytree("./roberta_reward_model_epoch2", "./roberta_reward_model_FINAL")

'./roberta_reward_model_FINAL'